# Chat Completion API로 프롬프트 엔지니어링 실습

Chat Completion API는 system, user, assistant 역할을 가진 메시지 목록을 모델에 전달하고 생성 응답을 받는 인터페이스이다. system 메시지는 모델의 역할과 공통 규칙을, user 메시지는 현재 요청과 입력 데이터를, assistant 메시지는 대화 이력을 표현한다. API 호출은 프롬프트를 코드로 재사용하고 결과 형식을 자동 처리할 수 있게 해 준다.

프롬프트 엔지니어링은 단순히 질문을 길게 쓰는 작업이 아니다. 역할, 입력 경계, 처리 규칙, 출력 스키마를 분리해 모델이 따라야 할 계약을 명확히 하는 작업이다. 강점은 빠른 실험과 다양한 업무 적용이며, 한계는 모델 출력이 확률적이고 외부 API 비용·속도·보안 제약이 있다는 점이다. API 키는 코드나 노트북에 저장하지 않고 실행 환경의 비밀 저장소에서 읽어야 한다.

이번 실습은 API 클라이언트를 준비한 뒤 기사 제목 교정, 상담형 응답, 레시피 제안, JSON 면접 질문 생성에 같은 메시지 구조가 어떻게 재사용되는지 확인한다. 외부 API를 호출하는 셀은 키·네트워크·모델 권한이 준비된 환경에서만 실행하고, 응답 내용은 업무 규칙과 JSON 파싱 결과로 다시 검증한다.


### OpenAI Python SDK 설치

이 셀은 Chat Completion API를 호출하기 위한 `openai` 패키지를 설치한다. 설치가 끝나면 이후 셀의 `OpenAI` 클래스를 가져올 수 있다. 패키지 버전과 인터넷 연결이 필요한 환경 준비 단계이므로, 이미 설치된 수업 환경에서는 실행 결과가 달라질 수 있다.


In [ ]:
# %pip install -U openai python-dotenv

## PyCharm 환경 설정

1. PyCharm에서 `08_llm` 폴더를 프로젝트로 연다.
2. PyCharm Terminal에서 `python -m pip install openai python-dotenv`를 실행한다.
3. `08_llm/.env` 파일에 `OPENAI_API_KEY`를 저장한다.
4. 키 값은 코드셀, 출력, Git 또는 공유 파일에 포함하지 않는다.


### PyCharm 프로젝트의 `.env` 설정 불러오기

`.env`는 API 키와 실행 설정을 노트북 코드에서 분리하는 로컬 파일이다. `find_dotenv(usecwd=True)`는 현재 Jupyter 작업 폴더부터 상위 폴더로 이동하며 `08_llm/.env`를 찾고, `load_dotenv()`는 그 값을 현재 커널의 환경 변수로 불러온다.

이 셀은 `OPENAI_API_KEY` 변수가 준비되었는지만 검사하고 실제 값은 출력하지 않는다. 이후 OpenAI SDK와 연동 라이브러리는 환경 변수를 자동으로 사용한다.


In [1]:

import os

from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError(
        "08_llm 프로젝트 최상위에 .env 파일을 만든 뒤 현재 셀을 다시 실행하세요."
    )

load_dotenv(dotenv_path, override=False)

required_env_vars = [
    "OPENAI_API_KEY",
]

missing_env_vars = [name for name in required_env_vars if not os.getenv(name)]
if missing_env_vars:
    missing_names = ", ".join(missing_env_vars)
    raise RuntimeError(f".env 파일의 다음 변수를 확인하세요: {missing_names}")

print("환경 변수 준비 완료")

환경 변수 준비 완료


### API 클라이언트 초기화

`OpenAI` 객체는 이후 모든 API 요청의 진입점이다. 앞 셀에서 읽은 키를 `api_key`에 전달해 `client`에 저장하고, 뒤의 프롬프트 함수들이 이 클라이언트를 재사용한다. 객체 생성은 네트워크 요청 자체가 아니지만, 실제 호출은 유효한 키와 사용 권한을 요구한다.


In [2]:
from openai import OpenAI

client = OpenAI()

### 가장 작은 Chat Completion 요청 만들기

이 셀은 `messages` 목록 안에 system과 user 메시지를 넣어 모델에 한 번 요청한다. `model`은 사용할 모델 이름, `temperature`는 생성 다양성, `max_completion_tokens`는 생성 길이 상한을 뜻한다. 응답 객체는 다음 셀에서 `choices[0].message.content`로 꺼낸다. 외부 API 호출이므로 실행하면 인증·네트워크·모델 사용 가능 여부를 함께 확인한다.


In [6]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",

    # 전달할 대화
    messages=[
        {
            "role": "system",  # 모델이 대화 전체에서 따를 역할과 규칙 작성
            "content": [
                {
                    "type": "text",
                    "text": "너는 아주 친절하고, 많은 도움을 주는 챗봇이야"
                }
            ]
        },
        {
            "role": "user",  # 사용자가 전달할 실제 입력 값
            "content": [
                {
                    "type": "text",
                    "text": "안녕~ 내 이름은 아이유야~"
                }
            ]
        }
    ],

    # 응답 형식 지정 "text", "json"
    response_format={"type": "text"},
    temperature=1,  # 온도 값이 높을수록 후보 토큰 수가 많아져 무작위성 증가
    max_completion_tokens=2048,  # 텍스트 생성 상한
    top_p=1,  # 누적 확률 범위 제한 X
    frequency_penalty=0,  # 같은 표현 반복 불이익 점수
    presence_penalty=0,  # 이미 등장한 주제를 다시 선택하는 경우 불이익 점수
)

### 응답 객체에서 생성 텍스트 추출하기

응답의 `choices`는 생성 후보 목록이며, 이 예제는 첫 번째 후보의 `message.content`를 표시한다. 후보가 비어 있거나 API 호출이 실패하면 이 접근은 불가능하므로, 앞 셀의 정상 응답 여부를 먼저 확인해야 한다. 출력 텍스트는 모델의 생성물이지 검증된 사실이 아니므로 업무 규칙과 원문을 대조한다.


In [7]:
print(response.choices[0].message.content)

안녕, 아이유! 만나서 정말 반가워~ 오늘 어떻게 지내고 있어? 도움이 필요하면 언제든 말해줘! 😊


## 프롬프팅의 기본구성

https://www.deeplearning.ai/short-courses/chatgpt-prompt-engineering-for-developers/

1. Instruction 지시사항
2. Context 문맥
3. Input Data/Example 입력/예시
4. Output Indicator 출력지시

## 기사 제목 교정

- 기자들이 송고한 기사에서 제목을 추출하고, 표현 조정
- 프랑스AFP 속보시스템에서 도입되어 사용


### 기사 제목 교정 프롬프트 실행

이 셀은 역할과 교정 규칙을 `system_message`에, 실제 기사 제목을 `user_message`에 분리해 전달한다. 예시와 출력 형식은 모델이 두 제목을 정해진 구조로 반환하도록 돕는다. `print` 결과에서 비속어 완화, 핵심 정보 보존, 지정된 두 줄 형식이 모두 지켜졌는지 확인한다. API 응답은 실행 환경에 따라 달라지며 결과를 실제 관찰한 값처럼 미리 단정하지 않는다.


In [9]:
# 교정이 필요한 제목
title_before = '테이의 FM 개꿀 라디오 방송에 주목해주세요.'
# title_before = '졸라 빡센 작업으로 끼니 거르기 일쑤인 노동자들의 애환'

# system 지시사항
# system 메시지에는 모든 제목에 공통으로 적용할 역할, 절차, 출력 형식, 예시를 넣는다.
system_message = """
기사들이 송고한 제목에서 맞춤법, 문법, 의미, 어조등에 있어서 교정작업을 수행해 주세요.

- 기사 제목이 명확하고 주제와 잘 맞도록 조정하세요.
- 독자의 관심을 끌 수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.
- 어조가 지나치게 감정적이거나 부정적인 경우 표현을 완화하거나, 중립적인 어조로 수정하세요.
- 비속어가 포함되어 있는 경우, 비속어를 반드시 제거하고, 의미를 적절히 유지하도록 제목을 교정하세요.

### Steps ###
1. 기사제목을 읽고 주요내용을 이해하세요.
2. 제목이 전달하고자하는 메세지를 명확하게 반영하는지 검토하세요.
3. 맞춤법, 문법, 의미 전달의 정확성등을 점검하고 적절히 수정하세요.
4. 제목이 자연스럽고, 독자에게 매력적으로 다가갈수 있는지 점검하고 매우 간결하게 정리하세요.


### Output Format ###
기사 원래 제목과 교정된 제목을 다음 형식으로 제공하세요.

- 원래 제목: [기사 원래 제목]
- 교정 제목: [교정 기사 제목]

### Examples ###
- 원래 제목: "어제 서울에서 큰 불이 나 수백명이 대피했다."
- 교정 제목: "서울 대형 화재, 수백명 대피"

- 원래 제목: "전기자동차 판매량 급감에 내연차회사들이 즐거워하는 중입니다."
- 교정 제목: "전기차 판매량 급감에 웃는 내연차회사들"

### Extra Instructions ###
- 제목이 너무 길면, 간결하게 줄이되, 핵심 메세지를 잃어버려서는 안됩니다.
- 지역명, 시간 등의 중요한 정보는 명확하게 유지하세요.
- 제목이 특정집단이나 대상에 대해 중립적이지 않을 경우, 그 표현을 완화하세요.

"""

# 이번 요청에서만 바뀌는 기사 제목을 user 메시지에 삽입
user_message = f"""
다음 기사제목을 교정해주세요.

제목: {title_before}
"""

response = client.chat.completions.create(
    model="gpt-4.1-mini",  # 제목 교정에 사용할 모델 ID이다.
    messages=[
        {
            "role": "system",  # 위에서 만든 공통 교정 규칙을 전달한다.
            "content": [
                {
                    "type": "text",
                    "text": system_message
                }
            ]
        },
        {
            "role": "user",  # 교정할 제목이 포함된 현재 요청을 전달한다.
            "content": [
                {
                    "type": "text",
                    "text": user_message
                }
            ]
        }
    ],
    response_format={
        "type": "text"  # 결과를 일반 문자열로 받는다.
    },
    temperature=1,  # 표현의 다양성을 조절한다.
    max_completion_tokens=2048,  # 교정 결과가 사용할 수 있는 생성 토큰 상한이다.
    top_p=1,  # 후보 토큰의 누적 확률 범위를 제한하지 않는다.
    frequency_penalty=0,  # 동일 표현 반복에 대한 추가 패널티를 사용하지 않는다.
    presence_penalty=0  # 새로운 주제 사용을 강제로 유도하지 않는다.
)

# 응답 결과 확인
print(response.choices[0].message.content)

- 원래 제목: 졸라 빡센 작업으로 끼니 거르기 일쑤인 노동자들의 애환
- 교정 제목: 힘든 작업 환경에 끼니를 거르는 노동자들의 어려움


### 반복 요청을 제목 교정 함수로 리팩터링하기

`correct_news_title`은 제목, 모델 이름, `temperature`, `top_p`를 인자로 받아 같은 프롬프트 구조를 재사용한다. 함수의 반환값은 응답 텍스트이며, 호출한 쪽은 `output`에 저장해 출력하거나 후속 검증에 사용한다. 같은 제목을 여러 설정으로 비교할 때는 입력을 고정하고 생성 파라미터만 바꿔 형식 준수와 표현 차이를 비교한다.


In [11]:
def correct_news_title(title_before, model='gpt-4.1-mini', temperature=1, top_p=1):
    system_message = """
    기사들이 송고한 제목에서 맞춤법, 문법, 의미, 어조등에 있어서 교정작업을 수행해 주세요.

    - 기사 제목이 명확하고 주제와 잘 맞도록 조정하세요.
    - 독자의 관심을 끌 수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.
    - 어조가 지나치게 감정적이거나 부정적인 경우 표현을 완화하거나, 중립적인 어조로 수정하세요.
    - 비속어가 포함되어 있는 경우, 비속어를 반드시 제거하고, 의미를 적절히 유지하도록 제목을 교정하세요.

    ### Steps ###
    1. 기사제목을 읽고 주요내용을 이해하세요.
    2. 제목이 전달하고자하는 메세지를 명확하게 반영하는지 검토하세요.
    3. 맞춤법, 문법, 의미 전달의 정확성등을 점검하고 적절히 수정하세요.
    4. 제목이 자연스럽고, 독자에게 매력적으로 다가갈수 있는지 점검하고 매우 간결하게 정리하세요.


    ### Output Format ###
    기사 원래 제목과 교정된 제목을 다음 형식으로 제공하세요.

    - 원래 제목: [기사 원래 제목]
    - 교정 제목: [교정 기사 제목]

    ### Examples ###
    - 원래 제목: "어제 서울에서 큰 불이 나 수백명이 대피했다."
    - 교정 제목: "서울 대형 화재, 수백명 대피"

    - 원래 제목: "전기자동차 판매량 급감에 내연차회사들이 즐거워하는 중입니다."
    - 교정 제목: "전기차 판매량 급감에 웃는 내연차회사들"

    ### Extra Instructions ###
    - 제목이 너무 길면, 간결하게 줄이되, 핵심 메세지를 잃어버려서는 안됩니다.
    - 지역명, 시간 등의 중요한 정보는 명확하게 유지하세요.
    - 제목이 특정집단이나 대상에 대해 중립적이지 않을 경우, 그 표현을 완화하세요.

    """
    user_message = f"""
    다음 기사제목을 교정해주세요.

    제목: {title_before}
    """

    response = client.chat.completions.create(
        model=model,  # 호출자가 선택한 모델 ID이다.
        messages=[
            {
                "role": "system",
                "content": [
                    {
                    "type": "text",
                    "text": system_message
                    }
                ]
            },
            {
                "role": "user",  # 이번에 교정할 제목이다.
                "content": [
                    {
                    "type": "text",
                    "text": user_message
                    }
                ]
            }],
            response_format={
                "type": "text"
            },
        temperature=temperature,
        max_completion_tokens=2048,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0
    )
    # 호출부가 SDK 응답 구조를 몰라도 되도록 생성된 문자열만 반환한다.
    return response.choices[0].message.content

title_before = '주말 미친 폭우 예상, 모두들 무사하시길~'
output = correct_news_title(title_before)
print(output)

- 원래 제목: 주말 미친 폭우 예상, 모두들 무사하시길~
- 교정 제목: 주말 강한 폭우 예상, 안전에 유의하세요


## 연애코치 ReAct

### ReAct 형식의 상담 프롬프트 함수 정의

ReAct는 추론과 행동을 명시적으로 구분해 복잡한 작업을 단계적으로 수행하게 하는 프롬프팅 방식이다. 이 예제는 외부 도구 호출을 구현하지 않고 상황 분석·행동 계획·실행이라는 출력 구조를 요청한다. `dating_coach`는 사용자 고민 문자열을 받아 응답 텍스트를 돌려주며, 다음 두 셀이 서로 다른 입력에서 구조가 유지되는지 확인한다.


### 첫 번째 상담 입력으로 출력 형식 확인

정의한 `dating_coach`에 기념일 선물이라는 한 문장 입력을 전달한다. 출력에서 상황 분석, 행동 계획, 실행 항목이 구분되는지와 안전하지 않거나 과도하게 단정적인 조언이 없는지를 확인한다. 실제 결과는 모델과 실행 시점에 따라 달라진다.


### 두 번째 상담 입력으로 일반화 범위 확인

같은 함수를 다른 갈등 상황에 적용해 프롬프트 구조가 입력 변화에도 유지되는지 확인한다. 두 응답을 비교할 때는 문장 길이보다 상황에 맞는 행동 제안과 출력 형식 준수 여부를 기준으로 삼는다. 개인 관계 조언은 사실 판단이나 전문 상담을 대체하지 않는다는 한계도 함께 안내한다.


## 냉털마스터 ReAct
- 사용자는 냉장고에 남아있는 음식재료를 알려주면, LLM은 이를 바탕으로 어떤 음식을 만들지를 조언해준다.
- Reasoning/Action을 끌어낼수 있는 적절한 프롬프팅을 작성한다.


### 재료 기반 레시피 제안 함수 정의

이 함수는 재료 목록을 user 메시지에 넣고, 분석·계획·검증·최종 레시피 순서의 응답을 요청한다. 리스트인 `user_foods`는 f-string에서 문자열로 변환되어 프롬프트의 입력 맥락이 된다. 모델은 실제 식재료 상태나 알레르기를 알 수 없으므로, 생성한 레시피는 조리 전 안전성·보관 상태·알레르기 정보를 사용자가 다시 확인해야 한다.


### 재료 목록을 Markdown 응답으로 표시하기

`Markdown`과 `display`는 모델이 반환한 마크다운 형식의 레시피를 노트북에서 읽기 쉽게 렌더링한다. `user_foods`가 함수 입력이고 반환 문자열이 `Markdown`의 입력이 된다. 화면에서는 네 개의 출력 구역이 실제로 구분되는지와 재료 목록이 모두 반영되었는지 확인한다.


## 면접질문 생성 JSON 출력


### JSON 형식 면접 질문 생성 함수 정의

이 함수는 채용공고 문자열을 받아 hard skill과 soft skill 질문·답변을 담은 JSON 문자열을 요청한다. `response_format={"type": "json_object"}`는 JSON 객체 형식의 응답을 요구하지만, 실제 파싱 전에는 필수 키와 값의 타입을 검증해야 한다. 함수는 아직 API를 호출하지 않으며 다음 셀의 채용공고가 입력으로 전달될 때 호출된다.


### 지시문과 입력 데이터를 분리한 JSON 프롬프트

이 셀은 시스템 역할 설명과 사용자 쪽의 지시·채용공고를 분리해 같은 JSON 생성 함수를 다시 정의한다. 프롬프트 구성 요소를 분리하면 역할 규칙은 재사용하고 채용공고만 바꿀 수 있다. 두 함수 정의는 같은 이름을 사용하므로 이 셀을 실행하면 앞 셀의 정의를 덮어쓴다는 점을 확인한다.


### 채용공고를 입력으로 전달하고 원본 응답 확인

긴 `job_posting` 문자열은 모델이 근거로 삼을 입력 데이터이며, `output`은 함수가 반환한 JSON 문자열이다. 먼저 `type(output)`이 `str`인지와 내용이 JSON 객체처럼 보이는지 확인한 뒤 다음 셀에서 파싱한다. 실제 API 호출은 비용과 네트워크가 발생하므로 수업 환경의 키와 사용 한도를 확인한 뒤 실행한다.


### JSON 문자열을 Python 자료구조로 변환하기

`json.loads`는 JSON 문자열 `output`을 Python 딕셔너리로 변환한다. 변환 뒤 `hard_skill`과 `soft_skill` 키로 질문·답변 목록을 꺼내 후속 화면 표시, 파일 저장, 평가에 사용할 수 있다. JSON 문법이 깨지거나 필수 키가 없으면 오류가 나므로, API 응답 형식과 키 존재를 검증하는 것이 실무에서 중요하다.
